In [0]:
dbutils.widgets.text("environment", "dev")
env = dbutils.widgets.get("environment")
print(env)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {env}_banking.data_source.branch (
        branch_id STRING NOT NULL COMMENT 'Unique identifier for the bank branch',
        branch_name STRING COMMENT 'Name of the bank branch',
        city STRING COMMENT 'City where the branch is located',
        state STRING COMMENT 'State or province where the branch is located',
        region STRING COMMENT 'Geographic region to which the branch belongs',
        source_modified_at TIMESTAMP COMMENT 'Timestamp when the source record was last modified'
    )
    USING DELTA
    COMMENT 'Master data for bank branches';     
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.data_source.product (
    product_id STRING NOT NULL COMMENT 'Unique identifier for the banking product',
    product_name STRING COMMENT 'Name of the banking product',
    product_category STRING COMMENT 'Category or type of the banking product',
    annual_fee DECIMAL(10, 2) COMMENT 'Annual fee charged for the banking product'
)
USING DELTA
COMMENT 'Master data for banking products and associated annual fees';
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.data_source.customer_history (
    customer_id STRING NOT NULL COMMENT 'Unique identifier for the customer',
    full_name STRING COMMENT 'Full name of the customer',
    address_line_1 STRING COMMENT 'Primary address line of the customer',
    city STRING COMMENT 'City associated with the customer address',
    state STRING COMMENT 'State or province associated with the customer address',
    postal_code STRING COMMENT 'Postal or ZIP code associated with the customer address',
    customer_segment STRING COMMENT 'Business or customer segment assigned to the customer',
    kyc_status STRING COMMENT 'Know Your Customer (KYC) verification status',
    effective_from_date DATE NOT NULL COMMENT 'Date from which this version of the customer record is effective',
    source_modified_at TIMESTAMP COMMENT 'Timestamp when the source record was last modified',
    source_system STRING COMMENT 'Source system from which the customer record originated'
)
USING DELTA
COMMENT 'Historical customer records containing effective-dated customer attributes';
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.data_source.account (
    account_id STRING NOT NULL COMMENT 'Unique identifier for the account',
    primary_customer_id STRING COMMENT 'Identifier of the primary customer associated with the account',
    branch_id STRING COMMENT 'Identifier of the branch where the account is maintained',
    account_type STRING COMMENT 'Type or category of the account',
    account_status STRING COMMENT 'Current status of the account',
    open_date DATE COMMENT 'Date on which the account was opened',
    close_date DATE COMMENT 'Date on which the account was closed, if applicable',
    currency_code STRING COMMENT 'ISO currency code for the account',
    source_modified_at TIMESTAMP COMMENT 'Timestamp when the source record was last modified'
)
USING DELTA
COMMENT 'Bank account master data and account lifecycle information';
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.data_source.account_customer_history (
    account_id STRING NOT NULL COMMENT 'Unique identifier for the account',
    customer_id STRING NOT NULL COMMENT 'Unique identifier for the customer associated with the account',
    ownership_role STRING COMMENT 'Role of the customer in relation to the account',
    ownership_percent INT COMMENT 'Percentage of account ownership held by the customer',
    effective_from_date DATE NOT NULL COMMENT 'Date from which this customer-account relationship is effective',
    source_modified_at TIMESTAMP COMMENT 'Timestamp when the source relationship record was last modified',
    source_system STRING COMMENT 'Source system from which the relationship record originated'
)
USING DELTA
COMMENT 'Historical effective-dated relationships between customers and accounts';
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.data_source.transaction (
    transaction_id STRING NOT NULL COMMENT 'Unique identifier for the transaction',
    account_id STRING COMMENT 'Identifier of the account associated with the transaction',
    transaction_timestamp TIMESTAMP COMMENT 'Timestamp when the transaction occurred',
    transaction_date DATE COMMENT 'Business date of the transaction',
    transaction_type STRING COMMENT 'Type or classification of the transaction',
    transaction_amount DECIMAL(12, 2) COMMENT 'Monetary amount of the transaction',
    currency_code STRING COMMENT 'ISO currency code of the transaction amount',
    merchant_name STRING COMMENT 'Name of the merchant associated with the transaction',
    merchant_category STRING COMMENT 'Category of the merchant associated with the transaction',
    transaction_status STRING COMMENT 'Current processing status of the transaction',
    source_modified_at TIMESTAMP COMMENT 'Timestamp when the source transaction record was last modified',
    arrival_timestamp TIMESTAMP COMMENT 'Timestamp when the transaction record arrived in the data platform',
    is_late_arrival BOOLEAN COMMENT 'Indicates whether the transaction arrived after the expected processing window',
    source_system STRING COMMENT 'Source system from which the transaction originated'
)
USING DELTA
COMMENT 'Banking transaction records including transaction, merchant, processing, and arrival information';
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.data_source.transaction_line (
    transaction_id STRING NOT NULL COMMENT 'Identifier of the parent transaction',
    line_number INT NOT NULL COMMENT 'Sequential line number within the transaction',
    product_id STRING COMMENT 'Identifier of the banking product associated with the transaction line',
    quantity INT COMMENT 'Quantity associated with the transaction line',
    line_amount DECIMAL(12, 2) COMMENT 'Monetary amount for the transaction line'
)
USING DELTA
COMMENT 'Detailed line-level information associated with banking transactions';
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.data_source.daily_account_balance (
    balance_date DATE NOT NULL COMMENT 'Business date for the account balance snapshot',
    account_id STRING NOT NULL COMMENT 'Unique identifier for the account',
    currency_code STRING COMMENT 'ISO currency code of the account balance',
    available_balance_amount DECIMAL(12, 2) COMMENT 'Amount available for immediate use or withdrawal',
    ledger_balance_amount DECIMAL(12, 2) COMMENT 'Account balance recorded in the ledger',
    overdraft_limit_amount DECIMAL(12, 2) COMMENT 'Maximum overdraft amount available for the account',
    source_modified_at TIMESTAMP COMMENT 'Timestamp when the source balance record was last modified',
    source_system STRING COMMENT 'Source system from which the account balance originated'
)
USING DELTA
COMMENT 'Daily account balance snapshots including available, ledger, and overdraft balances';
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.data_source.batch_manifest (
    batch_id STRING NOT NULL COMMENT 'Unique identifier for the ingestion or processing batch',
    processing_date DATE COMMENT 'Business or processing date associated with the batch',
    load_type STRING COMMENT 'Type of data load performed by the batch',
    generated_at_utc TIMESTAMP COMMENT 'UTC timestamp when the batch manifest was generated',
    transaction_record_count BIGINT COMMENT 'Number of transaction records processed in the batch',
    late_arrival_record_count BIGINT COMMENT 'Number of late-arriving transaction records processed in the batch'
)
USING DELTA
COMMENT 'Metadata and control information for data ingestion and processing batches';
""")